In [2]:
import re
import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# 1. Load your data
file_path = 'cleaning.csv'
df = pd.read_csv(file_path)

# 2. Prepare tools and resources
# 2.1 Normalization dictionary: adjust entries as needed
normalization_dict = {
    'yg': 'yang',
    'dgn': 'dengan',
    'ga': 'tidak',
    'gak': 'tidak',
    'smg': 'semoga',
    'setuju bgt': 'sangat setuju',
    # tambahkan kosa kata lainnya di sini
}

# 2.2 Stopword list (Bahasa Indonesia)
stop_factory = StopWordRemoverFactory()
stopwords = set(stop_factory.get_stop_words())

# 2.3 Stemmer (Bahasa Indonesia)
stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

# 3. Define cleaning function
def clean_text(text):
    # 3.1 Case folding
    text = text.lower()

    # 3.2 Remove URLs, mentions, hashtags, numbers, and punctuation
    text = re.sub(r'https?://\S+|www\.\S+', '', text)        # URLs
    text = re.sub(r'@\w+', '', text)                          # mentions
    text = re.sub(r'#\w+', '', text)                          # hashtags
    text = re.sub(r'\d+', '', text)                           # numbers
    text = re.sub(r'[^\w\s]', ' ', text)                      # punctuation

    # 3.3 Tokenize
    tokens = text.split()

    # 3.4 Normalize tokens based on normalization_dict
    tokens = [normalization_dict.get(token, token) for token in tokens]

    # 3.5 Remove stopwords
    tokens = [tok for tok in tokens if tok not in stopwords]

    # 3.6 Stemming
    tokens = [stemmer.stem(tok) for tok in tokens]

    # 3.7 Rejoin into string
    return ' '.join(tokens)

# 4. Apply to your dataframe (store in new column)
df['text_cleaned'] = df['cleaned_text'].apply(clean_text)

# 5. (Optional) Save the cleaned data
output_path = 'cleaning.csv'
df.to_csv(output_path, index=False)

print(f"Cleaning complete. Cleaned data saved to {output_path}")


Cleaning complete. Cleaned data saved to cleaning.csv


In [3]:
import pandas as pd
from transformers import pipeline

# 1. Load data
df = pd.read_csv('cleaning.csv')

# Pastikan kolom text_cleaned sudah ada—jika belum, jalankan pipeline pembersihan dulu!

# 2. Inisialisasi HuggingFace sentiment pipeline (CPU-only)
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="indobenchmark/indobert-base-p1",    # model IndoBERT fine‑tuned untuk sentimen
    tokenizer="indobenchmark/indobert-base-p1",
    device=-1  # pakai CPU
)

# 3. Buat fungsi untuk memproses batch agar lebih efisien
def batch_sentiment(texts, batch_size=32):
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        preds = sentiment_analyzer(batch)
        results.extend(preds)
    return results

# 4. Jalankan analisis
texts = df['text_cleaned'].astype(str).tolist()
predictions = batch_sentiment(texts, batch_size=64)

# 5. Simpan ke dataframe
df['sentiment_label'] = [pred['label'] for pred in predictions]
df['sentiment_score'] = [pred['score'] for pred in predictions]

# 6. (Opsional) Simpan hasilnya
df.to_csv('cleaning_with_sentiment.csv', index=False)
print("Selesai: kolom sentiment_label & sentiment_score telah ditambahkan.")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cpu


Selesai: kolom sentiment_label & sentiment_score telah ditambahkan.


In [ ]:
import pandas as pd

# 1. Load hasil sentiment
df = pd.read_csv('cleaning_with_sentiment.csv')

# 2. Buat fungsi mapping
def map_to_category(row, pos_labels=None, neg_labels=None, neutral_threshold=0.1):
    label = row['sentiment_label']
    score = row['sentiment_score']
    
    # jika confidence sangat rendah, langsung anggap neutral
    if score < neutral_threshold:
        return 'neutral'
    
    # tentukan label apa yang dianggap positif/negatif
    # sesuaikan ini dengan urutan label model Anda
    if pos_labels is None:
        pos_labels = {'LABEL_3', 'LABEL_4'}  # misal model Anda: LABEL_4 paling positif
    if neg_labels is None:
        neg_labels = {'LABEL_0', 'LABEL_1'}  # LABEL_0 paling negatif
    
    if label in pos_labels:
        return 'positive'
    elif label in neg_labels:
        return 'negative'
    else:
        return 'neutral'

# 3. Terapkan ke DataFrame
df['sentiment_category'] = df.apply(map_to_category, axis=1)

# 4. Simpan hasil
df.to_csv('cleaning_with_sentiment_categorized.csv', index=False)
print("Kolom sentiment_category telah terisi dengan nilai positive/negative/neutral.")


Kolom sentiment_category telah terisi dengan nilai positive/negative/neutral.
